You have been hired by a rookie movie producer to help him decide what type of movies to produce and which actors to cast. You have to back your recommendations based on thorough analysis of the data he shared with you which has the list of 3000 movies and the corresponding details.

 As a data scientist, you have to first explore the data and check its sanity.

Further, you have to answer the following questions:
1. Which movie made the highest profit? Who were its producer and director? Identify the actors in that film.
2. This data has information about movies made in different languages. Which language has the highest average ROI (return on investment)?
3. Find out the unique genres of movies in this dataset.
4. Make a table of all the producers and directors of each movie. Find the top 3 producers who have produced movies with the highest average RoI?
5. Which actor has acted in the most number of movies? Deep dive into the movies, genres and profits corresponding to this actor.
6. Top 3 directors prefer which actors the most?

1. Which movie made the highest profit? Who were its producer and director? Identify the actors in that film.

In [1]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv('imdb_data.csv')

In [5]:
df['profit'] = df['revenue'] - df['budget']

In [6]:
row = df.loc[df['profit'].idxmax()]
print("Title : ", row['title'])

Title :  Furious 7


In [8]:
import ast
crew = ast.literal_eval(row['crew'])
director = [i['name'] for i in crew if i['job'] == 'Director']
print("Director : ", director)

Director :  ['James Wan']


In [9]:
producer = [i['name'] for i in crew if i['job'] == 'Producer']
print("Producer : ", producer)

Producer :  ['Vin Diesel', 'Neal H. Moritz', 'Michael Fottrell', 'Brandon Birtell']


In [10]:
cast = ast.literal_eval(row['cast'])
actors = [i['name'] for i in cast]
print("Actors : " , actors)

Actors :  ['Vin Diesel', 'Paul Walker', 'Dwayne Johnson', 'Michelle Rodriguez', 'Tyrese Gibson', 'Ludacris', 'Jordana Brewster', 'Djimon Hounsou', 'Tony Jaa', 'Ronda Rousey', 'Nathalie Emmanuel', 'Kurt Russell', 'Jason Statham', 'Sung Kang', 'Gal Gadot', 'Lucas Black', 'Elsa Pataky', 'Noel Gugliemi', 'John Brotherton', 'Luke Evans', 'Ali Fazal', 'Miller Kimsey', 'Charlie Kimsey', 'Eden Estrella', 'Gentry White', 'Iggy Azalea', 'Jon Lee Brody', 'Levy Tran', 'Anna Colwell', 'Viktor Hernandez', 'Steve Coulter', 'Robert Pralgo', 'Antwan Mills', 'J.J. Phillips', 'Jorge Ferragut', 'Sara Sohn', 'Benjamin Blankenship', 'D.J. Hapa', 'T-Pain', 'Brian Mahoney', 'Brittney Alger', 'Romeo Santos', 'Jocelin Donahue', 'Stephanie Langston', 'Jorge-Luis Pallo', 'Tego Calder√≥n', 'Nathalie Kelley', 'Shad Moss', 'Don Omar', 'Klement Tinaj', 'Caleb Walker', 'Cody Walker']


2. This data has information about movies made in different languages. Which language has the highest average ROI (return on investment)?

In [15]:
df['ROI'] = df['profit'] / df['budget']
valid_roi = df[(df['budget'] > 0) & (df['ROI'] != np.inf) & (df['ROI'].notnull())]
ans = valid_roi.groupby('original_language')['ROI'].mean().sort_values(ascending=False)
print(ans.head(1))

original_language
ko    381794.102281
Name: ROI, dtype: float64


3. Find out the unique genres of movies in this dataset.

In [17]:
genres_set = set()

for g in df['genres']:
    if pd.notna(g):
        genres = ast.literal_eval(g)
        for i in genres:
            genres_set.add(i['name'])

print(genres_set)

{'Adventure', 'Science Fiction', 'Romance', 'Thriller', 'Documentary', 'Action', 'Music', 'Foreign', 'TV Movie', 'Animation', 'Fantasy', 'Horror', 'History', 'Comedy', 'Western', 'Crime', 'War', 'Drama', 'Family', 'Mystery'}


4. Make a table of all the producers and directors of each movie. Find the top 3 producers who have produced movies with the highest average RoI?

In [21]:
rows = []

for _, r in valid_roi.iterrows():
    if pd.notna(r['crew']):
        crew = ast.literal_eval(r['crew'])
        producers = [i['name'] for i in crew if i['job']=='Producer']

        for p in producers:
            rows.append([p, r['ROI']])

temp = pd.DataFrame(rows, columns=['producer','ROI'])

print(temp.groupby('producer')['ROI'].mean().sort_values(ascending=False).head(3))

producer
Ji Sang-yong    4197475.625
Lee Eun-ha      4197475.625
Jang Jin        4197475.625
Name: ROI, dtype: float64


5. Which actor has acted in the most number of movies? Deep dive into the movies, genres and profits corresponding to this actor.

In [24]:
rows = []

for _, r in df.iterrows():
    if pd.notna(r['cast']):
        cast = ast.literal_eval(r['cast'])
        for i in cast:
            rows.append([i['name'], r['profit'], r['genres']])

temp = pd.DataFrame(rows, columns=['actor','profit','genres'])

top_actor = temp['actor'].value_counts().idxmax()
print("Top Actor:", top_actor)

actor_data = temp[temp['actor'] == top_actor]

print("Total Movies:", len(actor_data))
print("Avg Profit:", actor_data['profit'].mean())

# Deep dive into genres
actor_genres = set()
for g_list in actor_data['genres']:
    if pd.notna(g_list):
        genres = ast.literal_eval(g_list)
        for g in genres:
            actor_genres.add(g['name'])

print("Genres:", list(actor_genres))

Top Actor: Samuel L. Jackson
Total Movies: 30
Avg Profit: 226510055.66666666
Genres: ['Adventure', 'Comedy', 'Animation', 'Science Fiction', 'Fantasy', 'Romance', 'Western', 'Thriller', 'Horror', 'Crime', 'Documentary', 'Drama', 'Family', 'Mystery', 'Action']


6. Top 3 directors prefer which actors the most?

In [26]:
rows = []

for _, r in df.iterrows():
    if pd.notna(r['crew']) and pd.notna(r['cast']):
        crew = ast.literal_eval(r['crew'])
        cast = ast.literal_eval(r['cast'])

        director = [i['name'] for i in crew if i['job']=='Director']
        director = director[0] if director else None

        for actor in cast:
            rows.append([director, actor['name']])

temp = pd.DataFrame(rows, columns=['director','actor'])

top_directors = temp['director'].value_counts().head(3).index

for d in top_directors:
    print("\nDirector:", d)
    print(temp[temp['director']==d]['actor'].value_counts().head(3))


Director: Ron Howard
actor
Clint Howard    7
Rance Howard    6
James Ritz      4
Name: count, dtype: int64

Director: Steven Spielberg
actor
Harrison Ford    3
Pat Roach        2
Tom Hanks        2
Name: count, dtype: int64

Director: Todd Phillips
actor
Matt Walsh        3
Bradley Cooper    3
Juliette Lewis    3
Name: count, dtype: int64
